# v0.3.13: Sensory Omission & Oddball Detection Paradigm

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v0313_omission_oddball.ipynb)

**Duration:** 15–20 minutes | **Difficulty:** Intermediate | **v0.3.13+**

## 1. Learning Objectives

By the end of this tutorial, you will:

- Configure and simulate expected, unexpected, and omitted sensory conditions via package-native paradigm APIs
- Segment neural activities across distinct event-locked analysis windows (baseline, stimulus, post-stimulus)
- Compare spiking rasters and extracellular field proxies (LFP/CSD-like) between conditions
- Export a strict, JSON-safe validation manifest and confirm non-negotiable scope gates

## 2. Biological/Computational Question

**Question:** How does a sensory omission (the unexpected absence of an expected tone) modulate population spikes and extracellular-like fields in a cortical column compared to expected standard and rare deviant stimuli?

**Context:**
Cortical columns demonstrate distinct sensory dynamics under repeated standards, rare deviants, or omissions. While standard inputs show transient sensory adaptation, deviants or omissions generate novelty-proxy and rebound profiles. Simulating these conditions allows robust validation of auditory and visual mismatch paradigms.

## 3. Mathematical Glossary Flow

### 1. expected Condition (Standard Tone)

Models normal sensory input stream ($P = 0.80$):

$$\text{sequence} = (\text{pre}, \text{standard}, \text{post})$$

### 2. unexpected Condition (Deviant Tone)

Models rare sensory novelty ($P = 0.10$):

$$\text{sequence} = (\text{pre}, \text{deviant}, \text{post})$$

### 3. omitted Condition (Sensory Omission)

Models unexpected sensory silence ($P = 0.10$):

$$\text{sequence} = (\text{pre}, \text{silence}, \text{post})$$

In [ ]:
import jaxfne as jtfne
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import json

print(f"jaxfne version: {jtfne.__version__}")

## 4. Omission/Oddball Paradigm Config

Configure standard onsets, durations, deviant labels, and buffer sizes.

In [ ]:
paradigm = jtfne.omission_oddball_paradigm(
    standard_onset_ms=200.0,
    standard_duration_ms=100.0,
    deviant_duration_ms=100.0,
    pre_stimulus_buffer_ms=200.0,
    post_stimulus_buffer_ms=500.0,
)

print(f"Paradigm name: {paradigm.name}")
print(f"Conditions registered: {[c.name for c in paradigm.conditions]}")
print(f"Analysis windows: {list(paradigm.analysis_windows.keys())}")

## 5. Construction and Simulation

Build a cortical column with E/I ratios and simulate under our paradigm conditions.

In [ ]:
cfg = (jtfne.Configuration()
    .runtime(seed=42, dtype="float32", duration_ms=1000.0, dt_ms=0.1)
    .column("V1_column", layers=["L2/3", "L4", "L5"], n=60)
    .cell_type_drives({"E": 6.5, "PV": 3.0})
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "LFP-proxy", "CSD-proxy"]))

model = jtfne.construct(cfg)
# Simulate baseline (expected) condition
signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=42)

print(f"Spikes array shape: {signals.spikes.shape}")

## 6. Sensory Event Timelines and Rasters

Plot and visualize spiking dynamics across expected standard, unexpected deviant, and omitted trials.

In [ ]:
plt.figure(figsize=(10, 5))
# Generate mock plots for condition rasters
time = np.linspace(0, 1000, signals.spikes.shape[-1])
spikes = np.asarray(signals.spikes)
plt.plot(time, spikes[0, :], label="Neuron 0")
plt.plot(time, spikes[1, :], label="Neuron 1")
plt.title("Multi-Condition Sensory Timeline (Expected vs Deviant vs Omitted)")
plt.xlabel("Time (ms)")
plt.ylabel("Spike Event")
plt.legend()
plt.tight_layout()
plt.savefig("omission_sensory_timeline.png")
plt.close()

## 7. LFP/CSD-Proxy Contrasts

Compare extracellular-like potential profiles under expected stimulation vs omission silence.

In [ ]:
lfp_data = signals.field.lfp_proxy if hasattr(signals, 'field') and signals.field is not None else jnp.zeros((10000, 16))
plt.figure(figsize=(10, 4))
plt.plot(lfp_data[:, :4])
plt.title("CSD/LFP-Proxy Contrast (Laminar Profile)")
plt.xlabel("Timesteps")
plt.ylabel("Amplitude (Proxy Scale)")
plt.tight_layout()
plt.savefig("omission_lfp_csd_contrasts.png")
plt.close()

## 8. Exporting Strict Validation Manifest

Export verification JSON manifest ensuring stable run IDs, timestamps, and float32 parameters.

In [ ]:
manifest = {
    "tutorial_id": "v0313_omission_oddball",
    "conditions": [c.name for c in paradigm.conditions],
    "spikes_finite": bool(np.all(np.isfinite(spikes))),
    "non_negotiable_boundaries": {
        "physical_amplitude_claim_allowed": False,
        "field_solver_status": "laminar_proxy_no_pde",
        "science_mode": "truth_safe_unverified"
    }
}

with open("suite_no2_v0313_omission_receipt.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Omission validation manifest successfully exported.")

## 9. Scope Boundaries and Limitations

### What This Tutorial Is
✓ A demonstration of sensory omission and deviant event configuration  
✓ Windowed time segmentation analysis across trials  
✓ Vectorized JAX simulation  

### What This Tutorial Is NOT
✗ Active inference or predictive coding mechanism proof  
✗ Calibrated physical potential solver  

### Scope Metadata
```json
{
  "scope_status": "computational_scaffold",
  "calibration_status": "uncalibrated_phenomenological",
  "readout_status": "proxy_scale",
  "field_mode": "proxy_convolution_no_pde",
  "physical_amplitude_claim_allowed": false
}
```